# 最新の33組で学習し直す

手持ち測定を前提に、前に測った3×3の9組と6×4の24組を使います。
入力はすでに0〜1のNN入力なので、そのまま使います。
正解RGBは、提供された元Notebookに保存された正解画像の均一な色領域から復元した値です。
元PNGとの数値照合は未実施で、復元元の表示画像と出典はリポジトリの `retraining/data` に保存しています。

**上から順番に実行します。** Python環境に必要なのはNumPyとPyTorchです。
未導入なら別セルで `%pip install numpy torch` を実行し、必要に応じてカーネルを再起動してください。

計画のH=40・50・60に加え、H=100を追加比較します。H=100は計画の3条件とは区別します。
全モデルをseed=42、50,000 epoch、Adam lr=0.01、重みのみL1係数1e-6で学習します。
最終epochの重みを固定し、検証・最終評価のデータで再学習やcheckpoint選択はしません。

同じ0入力に異なる4色が対応するため、この33組全体のRGB8 MSEには約70.01の下限があります。
3×3・6×4・全33組のMSEを分けて表示します。これは保存入力でのPC上の学習誤差です。
実機で新たに読み取った画像のMSEや、独立した最終評価の精度とは区別してください。


In [ ]:
"""Re-train Dense FP32 models on the newly measured 33 learning inputs."""

import argparse
import csv
import hashlib
import json
from pathlib import Path
import platform
import tempfile
import time

import numpy as np
import torch
from torch import nn


class ColorNet(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.model = nn.Sequential(nn.Linear(3, hidden_size), nn.ReLU(),
                                   nn.Linear(hidden_size, hidden_size), nn.ReLU(),
                                   nn.Linear(hidden_size, 3))

    def forward(self, inputs):
        return self.model(inputs)


def load_training(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as f:
        records = list(csv.DictReader(f))
    if len(records) != 33 or [int(r["sample_id"]) for r in records] != list(range(1, 34)):
        raise ValueError("The current training dataset must contain ordered sample IDs 1..33.")
    x = np.array([[r[f"input_{c}"] for c in "rgb"] for r in records], dtype=np.float32)
    y8 = np.array([[r[f"target_{c}8"] for c in "rgb"] for r in records], dtype=np.float64)
    if not np.isfinite(x).all() or np.any((x < 0) | (x > 1)):
        raise ValueError("NN input RGB must already be normalized to 0..1.")
    if not np.isfinite(y8).all() or np.any((y8 < 0) | (y8 > 255)) or np.any(y8 != np.floor(y8)):
        raise ValueError("Reference RGB must be integer 0..255 values.")
    charts = [r["chart_id"] for r in records]
    if charts != ["train_3x3"] * 9 + ["train_6x4"] * 24:
        raise ValueError("Expected 9 row-major 3x3 samples then 24 row-major 6x4 samples.")
    return x, y8


def model_arrays(model):
    return [(layer.weight.detach().cpu().numpy().copy(), layer.bias.detach().cpu().numpy().copy())
            for layer in model.model if isinstance(layer, nn.Linear)]


def forward_fp32(layers, inputs):
    """Match bias-first, sequential FP32 accumulation in the Arduino sketch."""
    x = np.asarray(inputs, dtype=np.float32)
    for i, (weight, bias) in enumerate(layers):
        output = np.broadcast_to(bias, (len(x), len(bias))).copy()
        for column in range(weight.shape[1]):
            output += x[:, column:column + 1] * weight[:, column][None, :]
        x = np.maximum(output, np.float32(0)) if i < 2 else output
    return x


def rgb8(raw_output):
    return np.floor(np.clip(raw_output, 0, 1) * np.float32(255) + np.float32(.5)).astype(np.uint8)


def mse_rgb8(prediction, target):
    return float(np.mean((np.asarray(prediction, dtype=np.float64) - target) ** 2))


def float_literal(value):
    value = np.float32(value)
    text = format(float(value), ".9g")
    if "." not in text and "e" not in text.lower():
        text += ".0"
    if not np.isfinite(value) or np.float32(float(text)) != value:
        raise ValueError("Invalid FP32 parameter or failed literal round trip.")
    return text + "f"


def export_header(layers, hidden_size, dataset_hash, seed, epochs, path):
    lines = ["// Dense FP32, re-trained on the current 33 saved learning inputs.", "#pragma once",
             'static_assert(sizeof(float) == 4, "FP32 required");',
             f"constexpr int MODEL_HIDDEN_SIZE = {hidden_size};",
             f"constexpr int MODEL_TRAINING_SEED = {seed};",
             f"constexpr int MODEL_TRAINING_EPOCHS = {epochs};",
             f'const char MODEL_TRAINING_DATA_SHA256[] = "{dataset_hash}";']
    for layer, (weight, bias) in enumerate(layers, 1):
        for name, values in [(f"weight_{layer}", weight.ravel()), (f"bias_{layer}", bias)]:
            lines.append(f"const float {name}[] = {{")
            for start in range(0, len(values), 8):
                lines.append("  " + ", ".join(float_literal(v) for v in values[start:start + 8]) + ",")
            lines.append("};")
    Path(path).write_text("\n".join(lines) + "\n", encoding="utf-8")


def input_collision_bounds(x, y8):
    groups = {}
    for i, values in enumerate(x):
        groups.setdefault(tuple(values.tolist()), []).append(i)
    continuous = 0.0
    integer = 0.0
    collisions = []
    for indices in groups.values():
        if len(indices) > 1:
            targets = y8[indices]
            mean = targets.mean(axis=0)
            continuous += float(np.sum((targets - mean) ** 2))
            integer += float(np.sum((targets - np.floor(mean + .5)) ** 2))
            collisions.append({"sample_ids": [i + 1 for i in indices], "input_rgb": x[indices[0]].tolist(),
                               "target_rgb": targets.astype(int).tolist()})
    return {"groups": collisions, "minimum_continuous_rgb255_mse": continuous / y8.size,
            "minimum_rgb8_mse": integer / y8.size}


def evaluate_validation(layers, path):
    if path is None or not Path(path).exists():
        return None
    with Path(path).open(newline="") as f:
        records = list(csv.DictReader(f))
    x = np.array([[r[f"input_{c}"] for c in "rgb"] for r in records], dtype=np.float32)
    target = np.array([[r[f"target_{c}8"] for c in "rgb"] for r in records], dtype=np.float64)
    predicted = rgb8(forward_fp32(layers, x))
    readings = sorted({r["read_id"] for r in records})
    mses = [mse_rgb8(predicted[[r["read_id"] == read for r in records]],
                     target[[r["read_id"] == read for r in records]]) for read in readings]
    return {"role": "validation only; not used in training or checkpoint selection", "read_ids": readings,
            "mse_rgb8_by_read": mses, "mean_mse_rgb8": float(np.mean(mses)),
            "data_sha256": hashlib.sha256(Path(path).read_bytes()).hexdigest()}


def train_one(x, y8, hidden_size, output_dir, dataset_hash, epochs=50000, seed=42,
              learning_rate=.01, lambda_l1=1e-6, log_every=5000, validation_path=None):
    if hidden_size < 1 or epochs < 1:
        raise ValueError("Hidden size and epochs must be positive.")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    if (output_dir / "model_parameters.h").exists():
        raise FileExistsError(f"Frozen weights already exist in {output_dir}; use a new output directory.")
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(1)
    model = ColorNet(hidden_size).cpu().float()
    inputs = torch.from_numpy(np.asarray(x, dtype=np.float32))
    targets = torch.from_numpy((y8 / 255).astype(np.float32))
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    curve = []
    start = time.monotonic()

    def record(epoch):
        with torch.no_grad():
            mse = torch.mean((model(inputs) - targets) ** 2).item()
            l1 = sum(p.abs().sum().item() for name, p in model.named_parameters() if "weight" in name)
        curve.append({"epoch": epoch, "mse_normalized": mse, "l1_sum": l1, "loss": mse + lambda_l1 * l1})
        print(f"H={hidden_size} epoch={epoch}/{epochs} MSE(0..1)={mse:.8f}", flush=True)

    record(0)
    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        outputs = model(inputs)
        mse = torch.mean((outputs - targets) ** 2)
        l1 = sum(p.abs().sum() for name, p in model.named_parameters() if "weight" in name)
        loss = mse + lambda_l1 * l1
        if not torch.isfinite(loss):
            raise FloatingPointError("Nonfinite loss; no model exported.")
        loss.backward()
        optimizer.step()
        if epoch % log_every == 0 or epoch == epochs:
            record(epoch)
    model.eval()
    layers = model_arrays(model)
    raw = forward_fp32(layers, x)
    predicted = rgb8(raw)
    with torch.no_grad():
        torch_output = model(inputs).numpy()
    report = {
        "hidden_size": hidden_size, "epochs": epochs, "seed_numpy_and_torch": seed,
        "learning_rate": learning_rate, "lambda_l1_weights_only": lambda_l1, "optimizer": "Adam",
        "checkpoint_rule": "Final epoch; no validation-driven early stopping or checkpoint selection.",
        "device": "cpu", "threads": 1, "torch_version": str(torch.__version__), "numpy_version": np.__version__,
        "python_version": platform.python_version(), "training_data_sha256": dataset_hash,
        "training_rows": 33, "all_rows_retained": True,
        "pc_training_mse_normalized_torch": curve[-1]["mse_normalized"],
        "pc_training_mse_rgb255_raw": float(np.mean((raw.astype(np.float64) * 255 - y8) ** 2)),
        "pc_training_mse_rgb8_all33": mse_rgb8(predicted, y8),
        "pc_training_mse_rgb8_3x3": mse_rgb8(predicted[:9], y8[:9]),
        "pc_training_mse_rgb8_6x4": mse_rgb8(predicted[9:], y8[9:]),
        "torch_vs_sequential_fp32_max_abs_difference": float(np.max(np.abs(raw - torch_output))),
        "dense_model_array_bytes": int(sum(w.nbytes + b.nbytes for w, b in layers)),
        "input_collision_bounds": input_collision_bounds(x, y8),
        "saved_validation_evaluation": evaluate_validation(layers, validation_path),
        "wall_seconds": time.monotonic() - start,
        "interpretation": "PC fit on saved learning inputs, not newly measured device MSE. Hand-held variability and identical-input conflicting labels remain. Teacher RGB recovered from supplied Notebook reference displays; original PNG files not independently compared.",
        "experiment_role": "additional exploratory H=100 comparison" if hidden_size == 100 else "planned hidden size",
    }
    export_header(layers, hidden_size, dataset_hash, seed, epochs, output_dir / "model_parameters.h")
    report["model_header_sha256"] = hashlib.sha256((output_dir / "model_parameters.h").read_bytes()).hexdigest()
    torch.save({"hidden_size": hidden_size, "epoch": epochs, "seed": seed,
                "training_data_sha256": dataset_hash, "state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict()}, output_dir / "checkpoint.pt")
    arrays = {f"{kind}_{i}": value for i, (w, b) in enumerate(layers, 1)
              for kind, value in [("weight", w), ("bias", b)]}
    np.savez(output_dir / "model_arrays.npz", **arrays)
    with (output_dir / "learning_curve.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(curve[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(curve)
    with (output_dir / "training_predictions.csv").open("w", newline="") as f:
        writer = csv.writer(f, lineterminator="\n")
        writer.writerow(["sample_id", "predicted_r8", "predicted_g8", "predicted_b8", "target_r8", "target_g8", "target_b8"])
        for i, (prediction, truth) in enumerate(zip(predicted, y8), 1):
            writer.writerow([i, *prediction.tolist(), *truth.astype(int).tolist()])
    (output_dir / "summary.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"DONE H={hidden_size}: learning RGB8 MSE 3x3={report['pc_training_mse_rgb8_3x3']:.4f}, "
          f"6x4={report['pc_training_mse_rgb8_6x4']:.4f}, all33={report['pc_training_mse_rgb8_all33']:.4f}", flush=True)
    return report


def train_models(data_path, output_dir, hidden_sizes=(40, 50, 60, 100), epochs=50000, seed=42, validation_path=None):
    x, y8 = load_training(data_path)
    data_hash = hashlib.sha256(Path(data_path).read_bytes()).hexdigest()
    reports = [train_one(x, y8, h, Path(output_dir) / f"h{h}", data_hash, epochs=epochs,
                         seed=seed, validation_path=validation_path) for h in hidden_sizes]
    columns = ["hidden_size", "epochs", "seed_numpy_and_torch", "pc_training_mse_rgb8_3x3",
               "pc_training_mse_rgb8_6x4", "pc_training_mse_rgb8_all33", "dense_model_array_bytes"]
    with (Path(output_dir) / "models_summary.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns, lineterminator="\n", extrasaction="ignore")
        writer.writeheader()
        writer.writerows(reports)
    return reports



In [ ]:
# このNotebook単独で実行できるよう、測定済み33組を埋め込んでいます。
RUN_ROOT = Path(tempfile.mkdtemp(prefix="current33_training_", dir=Path.cwd()))
TRAIN_CSV = """sample_id,chart_id,row,column,raw_r,raw_g,raw_b,input_r,input_g,input_b,target_r8,target_g8,target_b8
1,train_3x3,0,0,201.80,42.40,35.00,0.8548,0.1775,0.1615,237,30,36
2,train_3x3,0,1,116.60,61.20,38.10,0.4381,0.3443,0.1972,105,189,69
3,train_3x3,0,2,43.90,33.00,40.00,0.0826,0.0941,0.2191,56,83,164
4,train_3x3,1,0,255.90,156.00,73.00,1.0000,1.0000,0.5998,246,235,20
5,train_3x3,1,1,83.90,89.00,73.40,0.2782,0.5909,0.6044,111,204,221
6,train_3x3,1,2,92.00,40.30,41.00,0.3178,0.1588,0.2307,184,81,158
7,train_3x3,2,0,42.00,38.00,33.00,0.0733,0.1384,0.1384,1,1,1
8,train_3x3,2,1,72.40,52.00,43.00,0.2220,0.2626,0.2537,128,127,127
9,train_3x3,2,2,212.00,126.80,95.00,0.9046,0.9264,0.8535,255,255,255
10,train_6x4,0,0,63.00,43.00,33.10,0.1238,0.0881,0.0116,116,79,66
11,train_6x4,0,1,131.90,62.00,42.00,0.7330,0.3270,0.1841,195,143,126
12,train_6x4,0,2,63.90,46.00,41.00,0.1317,0.1258,0.1647,91,121,156
13,train_6x4,0,3,48.00,35.10,25.00,0.0000,0.0000,0.0000,91,108,64
14,train_6x4,0,4,72.70,48.00,46.30,0.2095,0.1509,0.2674,132,127,175
15,train_6x4,0,5,78.00,89.00,62.40,0.2564,0.6667,0.5795,96,189,173
16,train_6x4,1,0,142.30,70.20,58.40,0.8249,0.4302,0.5019,222,122,48
17,train_6x4,1,1,47.00,34.10,36.00,0.0000,0.0000,0.0678,69,91,167
18,train_6x4,1,2,134.00,32.10,26.00,0.7515,0.0000,0.0000,198,80,96
19,train_6x4,1,3,43.00,22.10,21.00,0.0000,0.0000,0.0000,93,59,105
20,train_6x4,1,4,79.30,61.00,37.00,0.2679,0.3145,0.0872,157,188,59
21,train_6x4,1,5,179.10,66.90,41.00,1.0000,0.3887,0.1647,228,161,40
22,train_6x4,2,0,36.90,28.20,30.30,0.0000,0.0000,0.0000,39,63,144
23,train_6x4,2,1,47.40,43.50,22.00,0.0000,0.0943,0.0000,60,148,70
24,train_6x4,2,2,91.80,22.10,17.00,0.3784,0.0000,0.0000,179,55,58
25,train_6x4,2,3,99.30,41.20,24.00,0.4447,0.0654,0.0000,234,198,28
26,train_6x4,2,4,90.90,39.00,39.00,0.3705,0.0377,0.1260,192,79,146
27,train_6x4,2,5,64.00,59.00,53.30,0.1326,0.2893,0.4031,1,133,165
28,train_6x4,3,0,182.10,99.10,79.20,1.0000,0.7937,0.9050,241,241,234
29,train_6x4,3,1,131.70,65.00,50.80,0.7312,0.3648,0.3547,201,202,201
30,train_6x4,3,2,87.10,53.10,42.70,0.3369,0.2151,0.1977,161,162,162
31,train_6x4,3,3,60.10,35.90,28.00,0.0981,0.0000,0.0000,122,122,122
32,train_6x4,3,4,43.30,31.00,25.40,0.0000,0.0000,0.0000,84,85,86
33,train_6x4,3,5,59.00,37.10,27.00,0.0884,0.0138,0.0000,50,50,50
"""
DATA_PATH = RUN_ROOT / "current33.csv"
DATA_PATH.write_text(TRAIN_CSV, encoding="utf-8")
DATA_PROVENANCE = json.loads('{\n  "dataset_role": "training; distinct from validation/reference_image4 and final evaluation",\n  "rows": 33,\n  "chart_rows": {\n    "train_3x3": 9,\n    "train_6x4": 24\n  },\n  "input_scale": "Saved NN inputs already normalized/clipped to 0..1; no second division by 255.",\n  "acquisition": "Original scanner: A0, LED R8/G6/B7 via analogWrite 255/0, settling 10 ms, 10 ADC samples/channel with 2 ms spacing.",\n  "source_reading_files": [\n    {\n      "file_name": "scan_3x3_01.csv",\n      "sha256": "78b87baa70cd2a4db0d44a46063585aa5d5992952a73e410da997e6efe86a48d"\n    },\n    {\n      "file_name": "scan_6x4_01.csv",\n      "sha256": "581284646f6da4f7f47c254b7d730ac3b7936395680d13a09a37d050cbad6a7f"\n    }\n  ],\n  "source_notebook_file": "train_L1_normalization.ipynb",\n  "source_notebook_sha256": "e21bed594569ab5dd6200139cd51b436778f298e4864835f1f1ee11d78fe2f1c",\n  "target_source": "RGB sampled from uniform interiors of supplied Notebook saved reference-image displays after its 3x3/6x4 resizing.",\n  "original_png_files_verified_on_this_machine": false,\n  "reference_extraction": [\n    {\n      "chart_id": "train_3x3",\n      "display_png_sha256": "ff4155b9911715b7def59f387184a16171fc74194eb1ab77725fc7d7e0e5d16c",\n      "sample_centers": [\n        {\n          "row": 0,\n          "column": 0,\n          "display_pixel": [\n            116,\n            77\n          ]\n        },\n        {\n          "row": 0,\n          "column": 1,\n          "display_pixel": [\n            239,\n            77\n          ]\n        },\n        {\n          "row": 0,\n          "column": 2,\n          "display_pixel": [\n            362,\n            77\n          ]\n        },\n        {\n          "row": 1,\n          "column": 0,\n          "display_pixel": [\n            116,\n            200\n          ]\n        },\n        {\n          "row": 1,\n          "column": 1,\n          "display_pixel": [\n            239,\n            200\n          ]\n        },\n        {\n          "row": 1,\n          "column": 2,\n          "display_pixel": [\n            362,\n            200\n          ]\n        },\n        {\n          "row": 2,\n          "column": 0,\n          "display_pixel": [\n            116,\n            323\n          ]\n        },\n        {\n          "row": 2,\n          "column": 1,\n          "display_pixel": [\n            239,\n            323\n          ]\n        },\n        {\n          "row": 2,\n          "column": 2,\n          "display_pixel": [\n            362,\n            323\n          ]\n        }\n      ],\n      "uniform_patch_checked": "11x11 pixels at each center"\n    },\n    {\n      "chart_id": "train_6x4",\n      "display_png_sha256": "3a1eb015de0d0f1636fca15d7943259d06cf87d8749b27fc91ae792e92feee2c",\n      "sample_centers": [\n        {\n          "row": 0,\n          "column": 0,\n          "display_pixel": [\n            96,\n            57\n          ]\n        },\n        {\n          "row": 0,\n          "column": 1,\n          "display_pixel": [\n            178,\n            57\n          ]\n        },\n        {\n          "row": 0,\n          "column": 2,\n          "display_pixel": [\n            261,\n            57\n          ]\n        },\n        {\n          "row": 0,\n          "column": 3,\n          "display_pixel": [\n            344,\n            57\n          ]\n        },\n        {\n          "row": 0,\n          "column": 4,\n          "display_pixel": [\n            426,\n            57\n          ]\n        },\n        {\n          "row": 0,\n          "column": 5,\n          "display_pixel": [\n            509,\n            57\n          ]\n        },\n        {\n          "row": 1,\n          "column": 0,\n          "display_pixel": [\n            96,\n            139\n          ]\n        },\n        {\n          "row": 1,\n          "column": 1,\n          "display_pixel": [\n            178,\n            139\n          ]\n        },\n        {\n          "row": 1,\n          "column": 2,\n          "display_pixel": [\n            261,\n            139\n          ]\n        },\n        {\n          "row": 1,\n          "column": 3,\n          "display_pixel": [\n            344,\n            139\n          ]\n        },\n        {\n          "row": 1,\n          "column": 4,\n          "display_pixel": [\n            426,\n            139\n          ]\n        },\n        {\n          "row": 1,\n          "column": 5,\n          "display_pixel": [\n            509,\n            139\n          ]\n        },\n        {\n          "row": 2,\n          "column": 0,\n          "display_pixel": [\n            96,\n            222\n          ]\n        },\n        {\n          "row": 2,\n          "column": 1,\n          "display_pixel": [\n            178,\n            222\n          ]\n        },\n        {\n          "row": 2,\n          "column": 2,\n          "display_pixel": [\n            261,\n            222\n          ]\n        },\n        {\n          "row": 2,\n          "column": 3,\n          "display_pixel": [\n            344,\n            222\n          ]\n        },\n        {\n          "row": 2,\n          "column": 4,\n          "display_pixel": [\n            426,\n            222\n          ]\n        },\n        {\n          "row": 2,\n          "column": 5,\n          "display_pixel": [\n            509,\n            222\n          ]\n        },\n        {\n          "row": 3,\n          "column": 0,\n          "display_pixel": [\n            96,\n            304\n          ]\n        },\n        {\n          "row": 3,\n          "column": 1,\n          "display_pixel": [\n            178,\n            304\n          ]\n        },\n        {\n          "row": 3,\n          "column": 2,\n          "display_pixel": [\n            261,\n            304\n          ]\n        },\n        {\n          "row": 3,\n          "column": 3,\n          "display_pixel": [\n            344,\n            304\n          ]\n        },\n        {\n          "row": 3,\n          "column": 4,\n          "display_pixel": [\n            426,\n            304\n          ]\n        },\n        {\n          "row": 3,\n          "column": 5,\n          "display_pixel": [\n            509,\n            304\n          ]\n        }\n      ],\n      "uniform_patch_checked": "11x11 pixels at each center"\n    }\n  ],\n  "identical_input_groups": [\n    {\n      "sample_ids": [\n        13,\n        19,\n        22,\n        32\n      ],\n      "input_rgb": [\n        0.0,\n        0.0,\n        0.0\n      ],\n      "target_rgb": [\n        [\n          91,\n          108,\n          64\n        ],\n        [\n          93,\n          59,\n          105\n        ],\n        [\n          39,\n          63,\n          144\n        ],\n        [\n          84,\n          85,\n          86\n        ]\n      ]\n    }\n  ],\n  "minimum_possible_continuous_rgb255_training_mse_due_to_identical_inputs": 70.00252525252525,\n  "minimum_possible_rgb8_training_mse_due_to_identical_inputs": 70.01010101010101,\n  "constraints": [\n    "Hand-held measurement is accepted as the operating constraint; variability remains.",\n    "All 33 measured rows retained, including four all-zero inputs for different reference colors; no favorable-row filtering.",\n    "Reference RGB was recovered from Notebook figures; compare with original reference PNG values when available.",\n    "PC fit on saved training inputs is not a new physical reading and does not guarantee repeat-read or unseen-chart accuracy."\n  ]\n}\n')
(RUN_ROOT / "provenance.json").write_text(json.dumps(DATA_PROVENANCE, ensure_ascii=False, indent=2), encoding="utf-8")
print("今回の出力先:", RUN_ROOT)
x, target_rgb8 = load_training(DATA_PATH)
print("入力・正解の形:", x.shape, target_rgb8.shape)
print("同一入力によるRGB8 MSEの下限:", input_collision_bounds(x, target_rgb8)["minimum_rgb8_mse"])


In [ ]:
HIDDEN_SIZES = [40, 50, 60, 100]  # 100は追加比較。1条件だけなら例：[100]
EPOCHS = 50000
SEED = 42
reports = train_models(DATA_PATH, RUN_ROOT / "models", hidden_sizes=HIDDEN_SIZES,
                       epochs=EPOCHS, seed=SEED, validation_path=None)


In [ ]:
print("H | 学習3x3 MSE | 学習6x4 MSE | 学習33組 MSE | Dense配列byte")
for result in reports:
    print(f"{result['hidden_size']} | {result['pc_training_mse_rgb8_3x3']:.4f} | "
          f"{result['pc_training_mse_rgb8_6x4']:.4f} | {result['pc_training_mse_rgb8_all33']:.4f} | "
          f"{result['dense_model_array_bytes']}")
print("保存先:", RUN_ROOT / "models")


各Hのフォルダに `model_parameters.h`（const FP32）、`checkpoint.pt`、学習曲線、予測RGB、設定とMSEを保存します。

新しい重みでは、古い閾値0.2449や古いCSRモデルを流用せず、同じ検証入力で閾値選択からやり直します。
Arduino側は、ヘッダのHと推論コードのHを一致させて実機の固定入力チェックを行います。
手持ち測定のばらつきと同一入力の制約は、モデルを大きくしただけで解消したとは扱いません。
